# Python Type Hints (Type Annotations)

Type hints let you **annotate** variables and functions with their expected types.  
Python does NOT enforce them at runtime — but editors, linters, and tools like `mypy` use them to catch bugs early.

| Why use type hints? |
|---------------------|
| Catch bugs before running code |
| Self-documenting code — no need to read the body to know what a function expects |
| Better IDE autocomplete |
| Required in many professional codebases and PySpark UDFs |

## 1. Variable Annotations

In [ ]:
# Basic variable annotations — just documentation, not enforced
name: str = "Alice"
age: int = 30
salary: float = 95000.50
is_active: bool = True

print(name, age, salary, is_active)

# Python does NOT enforce this at runtime
age: int = "thirty"   # no error at runtime — but mypy would flag it
print("age is now:", age, type(age))

# Annotations are stored in __annotations__
x: int
y: str = "hello"
print("__annotations__:", __annotations__)

## 2. Function Annotations

Annotate **parameters** and the **return type** (after `->`).

In [ ]:
# Without type hints — unclear what types are expected
def add(a, b):
    return a + b

# With type hints — crystal clear
def add_typed(a: int, b: int) -> int:
    return a + b

def greet(name: str, times: int = 1) -> str:
    return (f"Hello, {name}! " * times).strip()

def is_adult(age: int) -> bool:
    return age >= 18

def nothing() -> None:   # function returns nothing
    print("I return None")

print(add_typed(3, 4))          # 7
print(greet("Alice", 3))        # Hello, Alice! Hello, Alice! Hello, Alice!
print(is_adult(20))             # True
nothing()

# View annotations
print("\nadd_typed annotations:", add_typed.__annotations__)

## 3. The `typing` Module — Complex Types

For lists, dicts, tuples, optional values, and more — use the `typing` module (Python 3.8 and below) or built-in generics (Python 3.9+).

In [ ]:
from typing import List, Dict, Tuple, Set, Optional, Union, Any

# List of strings
def get_names(employees: List[str]) -> List[str]:
    return [name.upper() for name in employees]

# Dict with string keys and int values
def get_score(scores: Dict[str, int], name: str) -> int:
    return scores.get(name, 0)

# Tuple with fixed types
def get_coordinates() -> Tuple[float, float]:
    return (40.7128, -74.0060)   # lat, lon of New York

# Optional — value can be the type OR None
def find_employee(emp_id: int) -> Optional[str]:
    db = {1: "Alice", 2: "Bob"}
    return db.get(emp_id)   # returns str or None

# Union — value can be one of several types
def process(value: Union[int, str]) -> str:
    return str(value).upper()

# Any — opt out of type checking (use sparingly)
def log_anything(value: Any) -> None:
    print(value)

print(get_names(["alice", "bob"]))              # ['ALICE', 'BOB']
print(get_score({"Alice": 90, "Bob": 75}, "Alice"))  # 90
print(get_coordinates())                        # (40.7128, -74.006)
print(find_employee(1))                         # Alice
print(find_employee(99))                        # None
print(process(42))                              # 42
print(process("hello"))                         # HELLO

## 4. Python 3.9+ — Simpler Syntax (No `typing` Import Needed)

From Python 3.9 onward you can use built-in types directly: `list[str]` instead of `List[str]`.

In [ ]:
import sys
print("Python version:", sys.version)

# Python 3.9+ syntax — use built-in generics directly
def process_records(records: list[dict[str, int]]) -> list[str]:
    return [str(r) for r in records]

def lookup(data: dict[str, list[int]], key: str) -> list[int]:
    return data.get(key, [])

# Python 3.10+ — use | instead of Union
def parse_id(value: int | str) -> int:   # Union[int, str]
    return int(value)

def find(name: str) -> str | None:       # Optional[str]
    db = {"Alice": "alice@corp.com"}
    return db.get(name)

print(lookup({"scores": [90, 85, 92]}, "scores"))
print(find("Alice"))
print(find("Bob"))

## 5. Type Aliases and `TypedDict`

Give complex types a readable name, or define expected dict structures.

In [ ]:
from typing import TypedDict

# Type alias — name a complex type for reuse
EmployeeList = list[dict[str, str | int]]

def get_active(employees: EmployeeList) -> EmployeeList:
    return [e for e in employees if e.get("active")]

# TypedDict — specify exactly what keys a dict must have
class Employee(TypedDict):
    name: str
    age: int
    department: str

def describe(emp: Employee) -> str:
    return f"{emp['name']} ({emp['age']}) — {emp['department']}"

alice: Employee = {"name": "Alice", "age": 30, "department": "Engineering"}
print(describe(alice))

# Use EmployeeList alias
staff = [
    {"name": "Alice", "active": True},
    {"name": "Bob",   "active": False},
]
print("Active:", get_active(staff))

## 6. Type Hints in PySpark UDFs

Type hints are **required** when using Pandas UDFs (vectorized UDFs) in PySpark — they tell Spark what type the function returns.

In [ ]:
# PySpark UDF example (illustration — requires active SparkSession to run)
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType, IntegerType
import pandas as pd

# Regular UDF — return type declared explicitly
@udf(returnType=StringType())
def to_upper(name: str) -> str:
    return name.upper() if name else None

# Pandas UDF — type hints in the function signature ARE the schema
@pandas_udf(IntegerType())
def double_salary(salary: pd.Series) -> pd.Series:
    return salary * 2

print("UDFs defined — type hints tell PySpark the input/output types.")
print("to_upper    : str -> str")
print("double_salary: pd.Series(int) -> pd.Series(int)")

## Quick Reference

```python
# Primitives
x: int = 5
y: float = 3.14
z: str = "hello"
flag: bool = True

# Collections (Python 3.9+)
names: list[str]
scores: dict[str, int]
pair: tuple[int, int]
tags: set[str]

# Optional / Union (Python 3.10+)
value: int | None        # Optional[int]
result: int | str        # Union[int, str]

# Functions
def add(a: int, b: int) -> int: ...
def greet(name: str = "World") -> str: ...
def save(data: dict[str, Any]) -> None: ...

# typing module (Python 3.8 and below)
from typing import List, Dict, Optional, Union, Any, Tuple
def func(items: List[str]) -> Optional[int]: ...
```

## Interview Questions

1. Does Python enforce type hints at runtime? What tools do?
2. What is the difference between `Optional[str]` and `str | None`?
3. What does `-> None` mean in a function signature?
4. When would you use `Any`? Why should it be used sparingly?
5. What is `TypedDict` and how does it differ from a regular `dict`?
6. Why are type hints required in PySpark Pandas UDFs?
7. What is the difference between `list[str]` (Python 3.9+) and `List[str]` from `typing`?

## Practice Exercises

**Easy:**
1. Annotate all variables and return types in a function `bmi(weight: ?, height: ?) -> ?` that computes BMI.
2. Write a function `full_name(first: str, last: str) -> str` with correct type hints.

**Medium:**
3. Write `top_scores(scores: dict[str, int], n: int) -> list[tuple[str, int]]` that returns the top N scorers.
4. Define a `TypedDict` called `Product` with `id: int`, `name: str`, `price: float`. Write a function that takes a list of Products and returns only those with price > 100.

**Advanced:**
5. Annotate a function `merge_configs(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]` that deep-merges two config dicts.
6. Write a PySpark-style function annotated with type hints that takes a `pd.Series` of strings and returns a `pd.Series` with each string reversed.